# 뉴엔AI 생활백서 PDF - CPU RAG 벡터화 실험

## Goal

`뉴엔AI_생활백서v3.0.pdf`를 페이지 출처가 보존된 청크와 벡터로 변환하고, 검색 평가 및 선택적 Gemma 4 답변 생성까지 재현합니다.

- 기본 실행: 추가 모델 없이 한국어 문자 n-gram 해시 벡터로 파이프라인과 검색을 검증
- 운영 후보: EmbeddingGemma 300M으로 임베딩 백엔드 교체
- 생성 연결: Docker의 `llama-server`가 준비되면 동일한 검색 결과로 Gemma 4 호출

기본 해시 벡터는 배포 구조와 데이터 흐름을 검증하기 위한 기준선이며 최종 의미 검색 모델이 아닙니다.

## Setup

### 1. 필요 패키지

현재 기본 실행에는 `pypdf`, `numpy`만 필요합니다.

EmbeddingGemma와 외부 벡터 DB를 활성화할 때 설치할 패키지:

```bash
pip install sentence-transformers torch qdrant-client httpx
```

EmbeddingGemma는 Hugging Face에서 접근 승인이 필요한 모델입니다. 모델 페이지에서 약관에 동의한 후,
토큰을 코드에 적지 말고 터미널의 `hf auth login` 또는 환경변수 `HF_TOKEN`으로 인증하세요.

CPU-only 환경이므로 CUDA 패키지는 설치하지 않습니다.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import re
import sys
import time
import unicodedata
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Sequence

import numpy as np
from pypdf import PdfReader

for dependency_path in (
    Path.cwd() / "work" / "embedding_deps",
    Path.cwd().parent / "work" / "embedding_deps",
):
    if dependency_path.exists() and str(dependency_path) not in sys.path:
        sys.path.insert(0, str(dependency_path))

### 2. 입력·출력 및 모델 설정

In [2]:
def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    if (current / "outputs").exists():
        return current
    if current.name == "outputs":
        return current.parent
    return current


@dataclass(frozen=True)
class Settings:
    pdf_path: Path = Path(r"C:\Users\newen\Downloads\회사생활가이드\뉴엔AI_생활백서v3.0.pdf")
    artifact_dir: Path = find_workspace_root() / "outputs" / "company_guide_rag"
    embedding_backend: str = "hash"  # hash | embeddinggemma
    embedding_model: str = "google/embeddinggemma-300M"
    embedding_dimension: int = 2048
    min_page_chars: int = 40
    max_chunk_chars: int = 900
    chunk_overlap_chars: int = 120
    top_k: int = 5
    use_llama_server: bool = True
    llama_base_url: str = "http://localhost:8080"
    llama_model: str = "google/gemma-4-E2B-it-qat-q4_0-gguf:Q4_0"


settings = Settings()
settings.artifact_dir.mkdir(parents=True, exist_ok=True)
assert settings.pdf_path.exists(), f"입력 PDF가 없습니다: {settings.pdf_path}"
assert settings.chunk_overlap_chars < settings.max_chunk_chars
print(json.dumps({**asdict(settings), "pdf_path": str(settings.pdf_path), "artifact_dir": str(settings.artifact_dir)}, ensure_ascii=False, indent=2))

{
  "pdf_path": "C:\\Users\\newen\\Downloads\\회사생활가이드\\뉴엔AI_생활백서v3.0.pdf",
  "artifact_dir": "C:\\Users\\newen\\Documents\\Codex\\2026-08-07\\w\\outputs\\company_guide_rag",
  "embedding_backend": "hash",
  "embedding_model": "google/embeddinggemma-300M",
  "embedding_dimension": 2048,
  "min_page_chars": 40,
  "max_chunk_chars": 900,
  "chunk_overlap_chars": 120,
  "top_k": 5,
  "use_llama_server": true,
  "llama_base_url": "http://localhost:8080",
  "llama_model": "google/gemma-4-E2B-it-qat-q4_0-gguf:Q4_0"
}


## Steps

### 3. PDF 페이지 추출과 품질 진단

페이지 번호와 문서명을 모든 청크에 남깁니다. 텍스트가 적은 페이지는 이미지·도식 중심일 가능성이 있으므로 `low_text_pages`에 기록합니다.

In [3]:
@dataclass(frozen=True)
class PageDocument:
    page: int
    title: str
    text: str
    char_count: int


def clean_pdf_text(raw_text: str, page_number: int) -> str:
    text = raw_text.replace("\x00", " ").replace("\u00ad", "")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(rf"(?m)^\s*{page_number}\s*$", " ", text)
    text = re.sub(r"NEWEN\s*AI", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s*([•▪])\s*", r"\n\1 ", text)
    text = re.sub(r"\s*([①②③④⑤⑥⑦⑧⑨⑩])\s*", r"\n\1 ", text)
    return text.strip()


def infer_title(text: str, page_number: int) -> str:
    candidates = [line.strip() for line in text.splitlines() if line.strip()]
    for line in candidates[:5]:
        line = re.sub(rf"\s*{page_number}\s*$", "", line).strip()
        if 3 <= len(line) <= 100:
            return line
    return f"페이지 {page_number}"


reader = PdfReader(settings.pdf_path)
page_documents: list[PageDocument] = []
for page_number, page in enumerate(reader.pages, start=1):
    raw_text = page.extract_text() or ""
    text = clean_pdf_text(raw_text, page_number)
    page_documents.append(PageDocument(page_number, infer_title(text, page_number), text, len(text)))

low_text_pages = [page.page for page in page_documents if page.char_count < settings.min_page_chars]
print(f"PDF 페이지={len(page_documents):,}, 추출 문자={sum(p.char_count for p in page_documents):,}")
print(f"텍스트 부족 페이지(<{settings.min_page_chars}자)={low_text_pages}")
for page in page_documents[:5]:
    print(f"p.{page.page} chars={page.char_count} title={page.title[:80]}")

PDF 페이지=55, 추출 문자=11,217
텍스트 부족 페이지(<40자)=[1, 14, 55]
p.1 chars=28 title=explores NEW and creates NEW
p.2 chars=54 title=목차조직도복지제도전자문서 작성 매뉴얼채용절차법인카드 회원가입 메뉴얼01.02.03.04.05.
p.3 chars=179 title=1. 조직도
p.4 chars=621 title=2. 복지제도
p.5 chars=453 title=2. 복지제도


### 4. 페이지 보존 청킹

이 문서는 슬라이드형 매뉴얼이라 페이지가 곧 하나의 의미 단위입니다. 900자를 넘는 페이지만 겹침을 두고 나누며, 검색 결과가 언제나 원본 페이지로 돌아갈 수 있게 메타데이터를 보존합니다.

In [4]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    document: str
    page: int
    title: str
    text: str
    char_start: int
    char_end: int


def split_page(page: PageDocument) -> list[Chunk]:
    if page.char_count < settings.min_page_chars:
        return []
    prefix = f"문서: {settings.pdf_path.stem}\n페이지: {page.page}\n제목: {page.title}\n"
    body = page.text
    chunks: list[Chunk] = []
    start = 0
    while start < len(body):
        end = min(start + settings.max_chunk_chars, len(body))
        if end < len(body):
            boundaries = [body.rfind(mark, start + settings.max_chunk_chars // 2, end) for mark in ("\n", ". ", "다. ")]
            boundary = max(boundaries)
            if boundary > start:
                end = boundary + 1
        content = body[start:end].strip()
        canonical = f"{settings.pdf_path.name}|{page.page}|{start}|{content}"
        chunk_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:24]
        chunks.append(Chunk(chunk_id, settings.pdf_path.name, page.page, page.title, prefix + content, start, end))
        if end >= len(body):
            break
        start = max(start + 1, end - settings.chunk_overlap_chars)
    return chunks


chunks = [chunk for page in page_documents for chunk in split_page(page)]
print(f"벡터화 대상 청크={len(chunks):,}, 대상 페이지={len({c.page for c in chunks}):,}")
print(f"청크 길이 최소/평균/최대={min(len(c.text) for c in chunks)}/{sum(map(lambda c: len(c.text), chunks))//len(chunks)}/{max(len(c.text) for c in chunks)}")
print(json.dumps(asdict(chunks[0]), ensure_ascii=False, indent=2)[:1000])

벡터화 대상 청크=52, 대상 페이지=52
청크 길이 최소/평균/최대=95/273/658
{
  "chunk_id": "4be02a27b10ad47acbf6984a",
  "document": "뉴엔AI_생활백서v3.0.pdf",
  "page": 2,
  "title": "목차조직도복지제도전자문서 작성 매뉴얼채용절차법인카드 회원가입 메뉴얼01.02.03.04.05.",
  "text": "문서: 뉴엔AI_생활백서v3.0\n페이지: 2\n제목: 목차조직도복지제도전자문서 작성 매뉴얼채용절차법인카드 회원가입 메뉴얼01.02.03.04.05.\n목차조직도복지제도전자문서 작성 매뉴얼채용절차법인카드 회원가입 메뉴얼01.02.03.04.05. 2",
  "char_start": 0,
  "char_end": 54
}


### 5. 벡터화 백엔드

- `hash`: 한글 문자 2~4-gram을 고정 차원에 투영합니다. 다운로드 없이 재현 가능하며 정확한 용어 검색 기준선에 적합합니다.
- `embeddinggemma`: 실제 의미 검색 후보입니다. Hugging Face 모델 접근 승인과 패키지 설치 후 설정만 변경합니다.

In [5]:
def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


def korean_ngrams(text: str) -> list[str]:
    compact = re.sub(r"\s+", "", unicodedata.normalize("NFKC", text).lower())
    grams: list[str] = []
    for width in (2, 3, 4):
        grams.extend(compact[i:i + width] for i in range(max(0, len(compact) - width + 1)))
    return grams or [compact]


def hash_encode(texts: Sequence[str], dimension: int) -> np.ndarray:
    matrix = np.zeros((len(texts), dimension), dtype=np.float32)
    for row, text in enumerate(texts):
        for token in korean_ngrams(text):
            digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
            column = int.from_bytes(digest, "little") % dimension
            matrix[row, column] += 1.0 if digest[0] % 2 == 0 else -1.0
    return normalize_rows(matrix)


class Embedder:
    def __init__(self) -> None:
        self.model = None
        if settings.embedding_backend == "embeddinggemma":
            if importlib.util.find_spec("sentence_transformers") is None:
                raise RuntimeError("sentence-transformers와 CPU용 torch를 설치하세요.")
            from sentence_transformers import SentenceTransformer
            try:
                self.model = SentenceTransformer(settings.embedding_model, device="cpu")
            except Exception as error:
                if "GatedRepoError" in type(error).__name__ or "gated repo" in str(error).lower():
                    raise RuntimeError(
                        "EmbeddingGemma 접근 승인이 필요합니다. Hugging Face 모델 페이지에서 약관에 동의하고 "
                        "`hf auth login` 또는 HF_TOKEN으로 인증한 뒤 다시 실행하세요."
                    ) from error
                raise

    def encode_documents(self, texts: Sequence[str]) -> np.ndarray:
        if self.model is None:
            return hash_encode(texts, settings.embedding_dimension)
        return np.asarray(self.model.encode(
            list(texts), prompt_name="document", normalize_embeddings=True,
            truncate_dim=256, batch_size=16, show_progress_bar=True,
        ), dtype=np.float32)

    def encode_query(self, query: str) -> np.ndarray:
        if self.model is None:
            return hash_encode([query], settings.embedding_dimension)[0]
        return np.asarray(self.model.encode(
            [query], prompt_name="query", normalize_embeddings=True, truncate_dim=256,
        )[0], dtype=np.float32)


started = time.perf_counter()
embedder = Embedder()
embeddings = embedder.encode_documents([chunk.text for chunk in chunks])
elapsed = time.perf_counter() - started
print(f"backend={settings.embedding_backend}, shape={embeddings.shape}, dtype={embeddings.dtype}, elapsed={elapsed:.3f}s")

backend=hash, shape=(52, 2048), dtype=float32, elapsed=0.020s


### 6. 재사용 가능한 벡터 산출물 저장

In [6]:
chunks_path = settings.artifact_dir / "chunks.jsonl"
vectors_path = settings.artifact_dir / f"embeddings_{settings.embedding_backend}.npy"
manifest_path = settings.artifact_dir / f"manifest_{settings.embedding_backend}.json"

with chunks_path.open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")
np.save(vectors_path, embeddings, allow_pickle=False)

manifest = {
    "source_pdf": str(settings.pdf_path),
    "source_sha256": hashlib.sha256(settings.pdf_path.read_bytes()).hexdigest(),
    "page_count": len(page_documents),
    "indexed_pages": sorted({chunk.page for chunk in chunks}),
    "low_text_pages": low_text_pages,
    "chunk_count": len(chunks),
    "embedding_backend": settings.embedding_backend,
    "embedding_model": settings.embedding_model if settings.embedding_backend == "embeddinggemma" else None,
    "embedding_shape": list(embeddings.shape),
    "embedding_dtype": str(embeddings.dtype),
    "normalized": True,
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print(f"저장 크기: chunks={chunks_path.stat().st_size:,}B vectors={vectors_path.stat().st_size:,}B")

{
  "source_pdf": "C:\\Users\\newen\\Downloads\\회사생활가이드\\뉴엔AI_생활백서v3.0.pdf",
  "source_sha256": "1749ab22e7bb10942cebe7d4fa70090d8cd1e6d62dd0468da74658c8906ee0ac",
  "page_count": 55,
  "indexed_pages": [
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54
  ],
  "low_text_pages": [
    1,
    14,
    55
  ],
  "chunk_count": 52,
  "embedding_backend": "hash",
  "embedding_model": null,
  "embedding_shape": [
    52,
    2048
  ],
  "embedding_dtype": "float32",
  "normalized": true
}
저장 크기: chunks=44,936B vectors=426,112B


### 7. 로컬 코사인 검색

In [7]:
@dataclass(frozen=True)
class SearchResult:
    rank: int
    score: float
    chunk: Chunk


def search(query: str, top_k: int | None = None) -> list[SearchResult]:
    limit = top_k or settings.top_k
    query_vector = embedder.encode_query(query)
    scores = embeddings @ query_vector
    top_indices = np.argsort(-scores)[:limit]
    return [SearchResult(rank, float(scores[index]), chunks[int(index)]) for rank, index in enumerate(top_indices, start=1)]


def show_results(query: str, top_k: int = 5) -> list[SearchResult]:
    results = search(query, top_k)
    print(f"Q: {query}")
    for result in results:
        preview = re.sub(r"\s+", " ", result.chunk.text)[:140]
        print(f"{result.rank}. score={result.score:.3f} p.{result.chunk.page} {result.chunk.title[:45]} | {preview}")
    return results


example_results = show_results("자기계발비를 신청할 때 어떤 증빙이 필요한가요?")

Q: 자기계발비를 신청할 때 어떤 증빙이 필요한가요?
1. score=0.202 p.33 3. 전자문서 작성방법 – 인사 – 복리후생_자기계발비품의서 | 문서: 뉴엔AI_생활백서v3.0 페이지: 33 제목: 3. 전자문서 작성방법 – 인사 – 복리후생_자기계발비품의서 3. 전자문서 작성방법 – 인사 – 복리후생_자기계발비품의서 ➁작성방법-녹색 테두리: 자동완성-붉은색 테두리: 선택 및 작성➂주의사항1)
2. score=0.191 p.9 2. 복지제도 | 문서: 뉴엔AI_생활백서v3.0 페이지: 9 제목: 2. 복지제도 2. 복지제도 • 교육훈련➀지원내용 : 업무상 필요한 교육비 관련 비용 전액 지원➁지원조건 : 자격증, 수료증 등을 취득자에 한해 지원 (환급 교육의 경우 환급조건 미달성 시 지원하지 
3. score=0.128 p.6 2. 복지제도 | 문서: 뉴엔AI_생활백서v3.0 페이지: 6 제목: 2. 복지제도 2. 복지제도 • 사내동호회➀개설방법 : 자유게시판 및 이메일을 통해 7인 이상의 회원 확보 [개설신청서] 작성 및 제출➁지원금액 : 1인 월2만원 (누적 불가)➂가입제한 : 동일인이 
4. score=0.122 p.8 페이지 8 | 문서: 뉴엔AI_생활백서v3.0 페이지: 8 제목: 페이지 8 2. 복지제도➁카카오T 지원 : 업무와 관련된 일로 이동할 때만 이용이 가능합니다.*이용범위- 미팅시간이 촉박하거나 택시가 다른 대중교통보다 빠를 때 (예시 : 대화역까지 택시 -> 교대역
5. score=0.107 p.10 2. 복지제도 | 문서: 뉴엔AI_생활백서v3.0 페이지: 10 제목: 2. 복지제도 2. 복지제도 • 호텔&리조트 예약➀예약자격 : 1년 이상 근속한 임직원(4대보험 가입일기준)➁총 예약가능 일수 : 회사 전체 150박➂예약기간 : 예약일 1~2개월 전에 리조트 게시


### 8. 검색 스모크 평가

정답 페이지가 상위 K개 결과에 포함되는지를 확인합니다. 해시 기준선은 용어가 겹치는 질문에는 강하지만 표현이 크게 달라지는 의미 검색에는 약합니다.

In [8]:
evaluation_cases = [
    ("자녀 학자금은 누가 지원받을 수 있나요?", {4}),
    ("사내 동호회 지원 금액은 얼마인가요?", {6}),
    ("업무용 카카오T는 언제 이용할 수 있나요?", {8}),
    ("호텔과 리조트 예약 자격을 알려주세요", {10}),
    ("그룹웨어에서 휴가 신청하는 방법", {11}),
    ("휴직원은 언제까지 작성해야 하나요?", {24}),
    ("자기계발비 품의서 증빙과 주의사항", {33}),
    ("구매 금액이 300만원을 초과하면 어떤 문서를 작성하나요?", {51}),
    ("회사 채용 절차는 어떻게 진행되나요?", {52}),
]

evaluation_rows = []
for query, expected_pages in evaluation_cases:
    results = search(query, settings.top_k)
    retrieved_pages = [result.chunk.page for result in results]
    hit = bool(expected_pages.intersection(retrieved_pages))
    reciprocal_rank = next((1 / result.rank for result in results if result.chunk.page in expected_pages), 0.0)
    evaluation_rows.append({
        "query": query, "expected_pages": sorted(expected_pages),
        "retrieved_pages": retrieved_pages, "hit_at_k": hit, "reciprocal_rank": reciprocal_rank,
    })

recall_at_k = sum(row["hit_at_k"] for row in evaluation_rows) / len(evaluation_rows)
mrr = sum(row["reciprocal_rank"] for row in evaluation_rows) / len(evaluation_rows)
print(f"Recall@{settings.top_k}={recall_at_k:.3f}, MRR@{settings.top_k}={mrr:.3f}")
for row in evaluation_rows:
    print(("PASS" if row["hit_at_k"] else "FAIL"), row["expected_pages"], row["retrieved_pages"], row["query"])

(settings.artifact_dir / f"retrieval_evaluation_{settings.embedding_backend}.json").write_text(
    json.dumps({"recall_at_k": recall_at_k, "mrr_at_k": mrr, "rows": evaluation_rows}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

Recall@5=1.000, MRR@5=0.778
PASS [4] [34, 4, 7, 6, 9] 자녀 학자금은 누가 지원받을 수 있나요?
PASS [6] [21, 6, 4, 9, 19] 사내 동호회 지원 금액은 얼마인가요?
PASS [8] [8, 7, 12, 4, 19] 업무용 카카오T는 언제 이용할 수 있나요?
PASS [10] [10, 52, 7, 9, 17] 호텔과 리조트 예약 자격을 알려주세요
PASS [11] [11, 32, 6, 30, 4] 그룹웨어에서 휴가 신청하는 방법
PASS [24] [24, 4, 48, 2, 51] 휴직원은 언제까지 작성해야 하나요?
PASS [33] [33, 9, 50, 51, 49] 자기계발비 품의서 증빙과 주의사항
PASS [51] [4, 51, 44, 50, 43] 구매 금액이 300만원을 초과하면 어떤 문서를 작성하나요?
PASS [52] [12, 52, 15, 13, 10] 회사 채용 절차는 어떻게 진행되나요?


### 9. 선택적 Gemma 4 답변 생성

In [9]:
SYSTEM_PROMPT = """당신은 뉴엔AI 사내 생활백서 질의응답 도우미입니다.
제공된 근거만 사용해 한국어로 답하세요.
근거에 없는 내용은 추측하지 말고 문서에서 확인할 수 없다고 답하세요.
각 핵심 문장 끝에 [p.페이지] 형식으로 출처를 표시하세요."""


def build_context(results: Sequence[SearchResult]) -> str:
    return "\n\n".join(
        f"[근거 {result.rank} | p.{result.chunk.page} | {result.chunk.title}]\n{result.chunk.text}"
        for result in results
    )


def answer(question: str) -> dict:
    results = search(question, settings.top_k)
    context = build_context(results)
    if not settings.use_llama_server:
        return {
            "answer": "llama-server가 비활성화되어 검색 근거만 반환했습니다.",
            "context": context,
            "citations": [{"page": r.chunk.page, "score": round(r.score, 4)} for r in results],
        }
    payload = json.dumps({
        "model": settings.llama_model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"질문: {question}\n\n{context}"},
        ],
        "reasoning_effort": "none",
        "temperature": 0.1,
        "max_tokens": 384,
        "stream": False,
    }, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        f"{settings.llama_base_url}/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        body = json.loads(response.read().decode("utf-8"))
    return {
        "answer": body["choices"][0]["message"]["content"],
        "citations": [{"page": r.chunk.page, "score": round(r.score, 4)} for r in results],
    }


answer_preview = answer("사내 동호회 지원 기준을 알려주세요")
print(answer_preview["answer"])
print(answer_preview["context"][:1200] if "context" in answer_preview else answer_preview["citations"])

사내 동호회 지원 기준에 대한 정보는 다음과 같습니다.

*   **지원 금액:** 참석한 동호회원 1인당 20,000원입니다 [p.21].
*   **지원 신청 방법:** 그룹웨어에서 전자결재를 통해 지출결의서를 작성하고 활동일지, 사진파일, 출석부를 첨부해야 합니다 [p.21].
*   **기타 필수 사항:** 활동일 기준 참석한 모든 동호회원이나 온 활동사진 증빙자료가 필수입니다 [p.21].
[{'page': 21, 'score': 0.2602}, {'page': 6, 'score': 0.1964}, {'page': 9, 'score': 0.1739}, {'page': 4, 'score': 0.1311}, {'page': 7, 'score': 0.1211}]


## Checks

### 10. 산출물 무결성 검사

원본 해시, 페이지 출처, 청크 ID, 벡터 정규화 및 저장 후 재로딩을 확인합니다.

In [10]:
reloaded_vectors = np.load(vectors_path, allow_pickle=False)
reloaded_chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line]
norms = np.linalg.norm(reloaded_vectors, axis=1)

assert len(reloaded_chunks) == len(chunks) == reloaded_vectors.shape[0]
assert len({chunk["chunk_id"] for chunk in reloaded_chunks}) == len(reloaded_chunks)
assert reloaded_vectors.dtype == np.float32
assert np.allclose(norms, 1.0, atol=1e-5)
assert all(1 <= chunk["page"] <= len(page_documents) for chunk in reloaded_chunks)
assert manifest["source_sha256"] == hashlib.sha256(settings.pdf_path.read_bytes()).hexdigest()
assert recall_at_k >= 0.5, "기준선 검색 품질이 지나치게 낮습니다. 전처리와 평가셋을 점검하세요."
print("PDF 추출 → 청킹 → 벡터화 → 저장 → 검색 무결성 검사 통과")

PDF 추출 → 청킹 → 벡터화 → 저장 → 검색 무결성 검사 통과


## Next Steps

1. `embedding_backend="embeddinggemma"`, `embedding_dimension=256`으로 바꿔 같은 평가셋을 재실행합니다.
2. 해시 기준선과 EmbeddingGemma의 Recall@5, MRR@5, 인덱싱 시간, RAM을 비교합니다.
3. 텍스트 부족 페이지는 원본 화면을 확인해 OCR 또는 비전 추출 필요 여부를 결정합니다.
4. Qdrant에 `chunk_id`, `page`, `title`, `document`, `text` payload와 벡터를 적재합니다.
5. Docker `llama-server`를 연결하고 근거 충실도 및 응답 지연시간을 측정합니다.

운영 전에는 인사·복지 규정의 최신성, 문서 접근권한, 개인정보 포함 여부를 별도로 검수해야 합니다.